# Neutron

In [4]:
import h5py

file_data = h5py.File('neutron/Ac225.h5', 'r')

# Menampilkan seluruh hierarki/struktur di dalam file
print("Struktur isi file:")
file_data.visit(print)

Struktur isi file:
Ac225
Ac225/energy
Ac225/energy/0K
Ac225/energy/1200K
Ac225/energy/2500K
Ac225/energy/250K
Ac225/energy/294K
Ac225/energy/600K
Ac225/energy/900K
Ac225/fission_energy_release
Ac225/fission_energy_release/betas
Ac225/fission_energy_release/delayed_neutrons
Ac225/fission_energy_release/delayed_photons
Ac225/fission_energy_release/fragments
Ac225/fission_energy_release/neutrinos
Ac225/fission_energy_release/prompt_neutrons
Ac225/fission_energy_release/prompt_photons
Ac225/fission_energy_release/q_prompt
Ac225/fission_energy_release/q_recoverable
Ac225/kTs
Ac225/kTs/1200K
Ac225/kTs/2500K
Ac225/kTs/250K
Ac225/kTs/294K
Ac225/kTs/600K
Ac225/kTs/900K
Ac225/reactions
Ac225/reactions/reaction_002
Ac225/reactions/reaction_002/0K
Ac225/reactions/reaction_002/0K/xs
Ac225/reactions/reaction_002/1200K
Ac225/reactions/reaction_002/1200K/xs
Ac225/reactions/reaction_002/2500K
Ac225/reactions/reaction_002/2500K/xs
Ac225/reactions/reaction_002/250K
Ac225/reactions/reaction_002/250K/xs
Ac

File `Ac225.h5` memiliki peta struktur secara garis besar:

```
Ac225.h5
├── (root attrs: filetype, version)
└── Ac225/                     [attrs: Z, A, metastable, atomic_weight_ratio, n_reaction]
    ├── energy/                {0K, 1200K, 2500K, ...}  → array grid energi (eV)
    ├── kTs/                   {1200K, 2500K, ...}       → skalar kT (eV)
    ├── fission_energy_release/
    │   └── fragments, prompt_neutrons, delayed_neutrons, prompt_photons,
    │       delayed_photons, betas, neutrinos, q_prompt, q_recoverable
    ├── reactions/
    │   └── reaction_002/      [attrs: mt, label, Q_value, center_of_mass, n_product, redundant]
    │       ├── 0K/xs          [attr: threshold_idx]
    │       ├── 1200K/xs       [attr: threshold_idx]
    │       ├── 2500K/xs       [attr: threshold_idx]
    │       └── product_j/     (opsional)
    └── total_nu/               [attrs: particle, emission_mode, decay_rate, n_distribution]
        └── yield
```

1. Level root file & level nuklida

    Pada root file, ada dua atribut: **filetype (string penanda jenis file)** dan **version (array 2-integer berisi versi major dan minor dari format data)**. Ini dipakai OpenMC untuk memastikan file kompatibel dengan versi solver yang dipakai.

    Pada file awal Ac225 sendiri:

    | Atribut | Tipe | Makna |
    |---|---|---|
    | Z | int | Nomor atom (jumlah proton) — untuk Ac225, Z=89 |
    | A | int | Nomor massa (jumlah nukleon); untuk elemen natural, A diisi 0 — untuk Ac225, A=225 |
    | metastable | int | Status metastabil: 0 = ground state, 1 = eksitasi pertama, dst. |
    | atomic_weight_ratio | double | Massa nuklida dalam satuan massa neutron (AWR = M_nuklida/M_neutron) |
    | n_reaction | int | Jumlah total reaksi yang tersedia dalam file |


2. **energy/** dan **kTs/** — grid energi dan temperatur

    **energy** adalah **group**, isinya satu dataset per temperatur (energy - temperatur, misalnya "0K", "1200K", "2500K"), dan tiap dataset berisi array energi dalam satuan eV tempat cross section ditabulasikan untuk temperatur itu. Ini adalah **union energy grid** ialah grid gabungan tempat semua reaksi pada nuklida itu dievaluasi, sehingga tidak perlu grid berbeda-beda per reaksi.

    Group **kTs/** dengan struktur nama yang sama, tapi isinya adalah nilai kT (Boltzmann constant × temperatur) dalam eV untuk setiap temperatur TTT dalam Kelvin. Formulanya:

    $$kT = k_B \times T$$

    dengan $k_B = 8.617333262 \times 10^{-5}\ \text{eV/K}$. Contoh: untuk 1200 K → $kT = 8.617333262\times10^{-5}\times1200 \approx 0.10341\ \text{eV}$; untuk 2500 K → $\approx 0.21543\ \text{eV}$.

    **Mengapa ada "0K"?** Untuk elastic scattering pada nuklida berat yang punya resonansi rendah, cross section elastic pada 0 K harus tersedia agar OpenMC bisa menangani model resonance elastic scattering (Doppler upscattering) dengan benar. 0 K bukan temperatur "fisik" biasa — ini cross section mentah (belum di-Doppler-broadening) yang dipakai algoritma DBRC (Doppler Broadening Rejection Correction) saat sampling tumbukan pada temperatur tinggi, bukan hasil interpolasi kT standar.


3. **fission_energy_release/** — pelepasan energi fisi

    Ini representasi kompak dari data ENDF MF=1, MT=458 — informasi untuk menghitung energi yang dibawa keluar dari reaksi fisi oleh masing-masing produk reaksi (fragmen inti, neutron, dst.) sebagai fungsi energi neutron datang.

    | Komponen | Makna fisis |
    |---|---|
    | fragments | Energi kinetik fragmen fisi sebagai fungsi energi neutron datang |
    | prompt_neutrons | Energi kinetik neutron prompt |
    | delayed_neutrons | Energi kinetik neutron delayed |
    | prompt_photons | Energi foton (gamma) prompt |
    | delayed_photons | Energi foton delayed (dari decay produk fisi) |
    | betas | Energi partikel beta dari decay produk fisi |
    | neutrinos | Energi yang dibawa neutrino (lolos, tidak terdeposit sebagai panas) |
    | q_prompt | Q-value fisi prompt: fragments + prompt_neutrons + prompt_photons − energi neutron datang |
    | q_recoverable | Q-value fisi recoverable: q_prompt + delayed_neutrons + delayed_photons + betas |

    Dalam konteks pustaka data OpenMC (seperti ENDF/B), atribut **type = b'Polynomial** menandakan bahwa data tersebut **tidak disimpan sebagai tabel panjang berisi titik-titik diskrit** (seperti data *cross-section*), melainkan disimpan sebagai **rumus matematika**.

    Secara spesifik, **Polynomial berarti nilai fisisnya dihitung menggunakan persamaan polinomial (suku banyak) terhadap energi neutron datang ($E$)**.
    **Ac225/fission_energy_release/betas**. Ini merepresentasikan **rata-rata energi yang dilepaskan oleh partikel beta peluruhan** setelah terjadinya reaksi fisi, sebagai fungsi dari energi neutron yang menabrak target.

    Bentuk umum persamaan polinomial di OpenMC adalah:

    $$f(E) = C_0 + C_1 E + C_2 E^2 + \dots + C_n E^n$$

    Di mana:

    * $E$ = Energi neutron datang (dalam eV)
    * $C_n$ = Koefisien polinomial

    **Mengapa disimpan dalam bentuk Polinomial?**
    Untuk parameter tertentu seperti pelepasan energi fisi (beta, gamma, neutrino), perubahannya terhadap energi neutron sangat halus dan terprediksi—seringkali linier. Menyimpan 3 angka koefisien ini jauh lebih hemat memori dan lebih cepat diproses oleh komputer dibandingkan menyimpan ratusan titik data pada grid energi.

    Sementara untuk Q, formulanya :

    $$Q_{prompt}(E) = E_{fragments}(E) + E_{prompt,n}(E) + E_{prompt,\gamma}(E) - E$$

    $$Q_{recoverable}(E) = Q_{prompt}(E) + E_{delayed,n}(E) + E_{delayed,\gamma}(E) + E_{\beta}(E)$$

    Suku $-E$ penting: nilai fragments, prompt_neutrons, prompt_photons yang dievaluasi ENDF sudah memasukkan kontribusi energi kinetik yang dibawa neutron datang ke inti majemuk. Supaya **q_prompt** benar-benar mencerminkan energi yang **dilepaskan oleh proses fisi itu sendiri** (bukan energi yang "dititipkan" neutron), energi datang itu dikurangkan. neutrinos sengaja tidak masuk q_recoverable karena neutrino lolos dari material tanpa berinteraksi — energinya tidak pernah terdeposit sebagai panas, makanya disebut "non-recoverable".

    Tiap dataset komponen di atas berupa **fungsi 1D** yang formatnya bisa **Polynomial atau Tabulated1D** (lihat bagian 6) — cek atribut/type pada masing-masing dataset untuk tahu formula mana yang berlaku.


4. **reactions/reaction_<mt>/** — cross section per reaksi

    **MT number**, kode identifikasi reaksi standar ENDF-6. Misal reaction_002 berarti MT=2, yaitu reaksi (n,elastic) — hamburan elastik. Beberapa MT umum lain :

    | MT | Label | Keterangan |
    |---|---|---|
    | 1 | (n,total) | Cross section total |
    | 2 | (n,elastic) | Hamburan elastik |
    | 4 | (n,inelastic) | Total hamburan inelastik |
    | 16 | (n,2n) | Emisi 2 neutron |
    | 18 | (n,fission) | Fisi total |
    | 51–91 | (n,n1)…(n,nc) | Inelastik ke level diskret/kontinum |
    | 101 | (n,disappear) | Total absorpsi (tanpa emisi neutron) |
    | 102 | (n,gamma) | Radiative capture |
    | 452/455/456 | ν̄ total/delayed/prompt | Yield neutron fisi |
    | 458 | — | Pelepasan energi fisi (lihat bagian D) |

    Atribut pada level reaction_002 (bukan per-temperatur):

    | Atribut | Tipe | Makna |
    |---|---|---|
    | mt | int | Nomor MT ENDF untuk reaksi ini — di sini 2 |
    | label | string | Nama reaksi — "(n,elastic)" |
    | Q_value | double | Nilai Q reaksi dalam eV — untuk elastic **selalu 0**, karena tidak ada perubahan identitas/keadaan inti (energi kinetik hanya dipindah antara neutron dan inti target) |
    | center_of_mass | int | Menunjukkan apakah kerangka acuan hamburan center-of-mass (1) atau laboratorium (0) |
    | n_product | int | Jumlah produk reaksi (untuk elastic biasanya 1, yaitu neutron yang terhambur) |
    | redundant | int | Menandakan apakah reaksi ini "redundant" — reaksi redundant (seperti MT=1 total, MT=4 total inelastic) cross section-nya adalah **penjumlahan** dari reaksi-reaksi parsial lain, direpresentasikan sebagai fungsi bertipe Sum (lihat bagian 6) |

    Di dalam tiap subgroup temperatur (0K/, 1200K/, 2500K/):

    - Dataset **xs**: nilai cross section yang ditabulasi terhadap grid energi nuklida pada temperatur tertentu dengan satuan barn [b].
    - Atribut **threshold_idx**: indeks pada grid energi tempat threshold reaksi berada untuk temperatur tertentu.

    Secara spesifik untuk **reaction_017** pada suhu 294K berdasarkan data, nilai **threshold_idx = 945** berarti **nilai cross-section ($xs$) pada indeks ke-0 berpasangan dengan nilai energi pada indeks ke-945** di dalam grid energi utama isotop **Ac225/energy**.

    Cara Membaca Pemetaannya:

    Jika Anda memiliki array energi utama bernama energy_grid (yang panjangnya sekitar 900-an atau lebih), maka:

    * Nilai $xs[0]$ (yaitu 0.0) $\rightarrow$ berpasangan dengan **energy_grid[945]** *(Inilah titik energi ambang batas fisisnya)*
    * Nilai $xs[1]$ (yaitu 0.0009225) $\rightarrow$ berpasangan dengan **energy_grid[946]**
    * Nilai $xs[2]$ (yaitu 0.00497209) $\rightarrow$ berpasangan dengan **energy_grid[947]**
    * ... dan seterusnya sampai data terakhir:
    * Nilai $xs[24]$ (yaitu 1.87087) $\rightarrow$ berpasangan dengan **energy_grid[945 + 24]** atau **energy_grid[969]**

    threshold_idx, merupakan teknik penghematan memori: array xs **tidak sepanjang penuh array energy, hanya mulai dari titik ambang**. Rekonstruksinya: cross section penuh dibentuk dengan memasangkan array xs terhadap energy[threshold_idx:] — yakni titik grid energi mulai dari indeks threshold_idx sampai akhir. Secara formula:

    $$\sigma(E_i) = \begin{cases} 0 & i < i_{th} \\ \text{xs}[\,i - i_{th}\,] & i \ge i_{th} \end{cases}$$

    Untuk MT=2 (elastic), biasanya $i_{th}=0$ karena hamburan elastik terjadi di semua energi (tak ada ambang).

    Jika suatu reaksi punya produk sekunder (misalnya (n,2n), fisi, dll.), akan ada subgroup **product_<j>/** dengan format
    - Reaction Products : atribut particle (jenis partikel), 
    - emission_mode (prompt/delayed/total), 
    - decay_rate (konstanta laju decay dalam per-detik, relevan untuk grup neutron delayed), 
    - n_distribution (jumlah distribusi sudut-energi); dataset yield (yield produk sebagai fungsi energi); dan 
    - group distribution_<k> untuk distribusi sudut-energi.

    Dalam struktur data HDF5 OpenMC, **product_0** dan **product_1** merepresentasikan **jenis partikel sekunder yang berbeda** yang dipancarkan dari satu reaksi nuklir yang sama.

    **reaction_016** adalah sandi untuk **reaksi $(n, 2n)$**. Ini adalah reaksi di mana satu neutron menabrak inti atom target, lalu menyebabkan **dua neutron** terlempar keluar.

    Berikut adalah rincian perbedaan keduanya:
    1. product_0 (Neutron)

        Secara *default*, produk pertama (product_0) dari reaksi yang diinisiasi oleh neutron hampir selalu adalah **Neutron** itu sendiri.

        * Meskipun reaksi $(n, 2n)$ menghasilkan 2 buah neutron, OpenMC tidak membaginya menjadi produk 0 dan produk 1.
        * Sebagai gantinya, OpenMC hanya mendefinisikan *jenis* partikelnya (neutron) di product_0, lalu mengatur jumlah partikel yang keluar di dalam *path* **product_0/yield**. (Jika Anda membongkar data yield ini, Anda akan menemukan nilai konstan 2.0).
        * *Path* **distribution_0/...** di bawahnya menyimpan informasi tentang bagaimana kedua neutron tersebut membagi energi (energy_out) dan ke arah mana mereka terpental (mu).

    2. product_1 (Foton / Sinar Gamma)

        Lalu, partikel apa yang menjadi product_1. Dalam mayoritas data nuklir, produk kedua ini adalah **Foton (Sinar Gamma)**.

        * Saat reaksi $(n, 2n)$ terjadi, inti atom target (misalnya $Ac^{225}$) kehilangan satu neutron dan berubah menjadi isotop lain ($Ac^{224}$). Inti sisa ini biasanya berada dalam keadaan tereksitasi (kelebihan energi).
        * Untuk kembali ke keadaan stabil, inti sisa tersebut memancarkan energi berlebihnya dalam bentuk radiasi gamma.
        * OpenMC harus memisahkan data foton ini ke dalam folder product_1 karena foton memiliki spektrum energi emisi dan distribusi sudut yang sifat fisisnya sama sekali berbeda dari hamburan neutron.


5. **total_nu/yield** — ν̄ (nu-bar) total

    Group total_nu adalah produk khusus yang mendefinisikan jumlah total neutron yang diproduksi dari fisi, diformat seperti "Reaction Product" — makanya atributnya mengikuti format di bagian 3 (particle = neutron, emission_mode = total, decay_rate, n_distribution), dan dataset **yield** berisi fungsi:

    $$\bar{\nu}_{total}(E)$$

    yaitu rata-rata jumlah neutron yang dilepaskan per reaksi fisi, sebagai fungsi energi neutron datang E (lagi-lagi berupa Polynomial atau Tabulated1D). Kuantitas ini dipakai bersama cross section fisi $\sigma_f(E)$ untuk menghitung laju produksi neutron fisi:

    $$\text{Produksi neutron} \propto \sigma_f(E)\cdot\bar{\nu}(E)\cdot\phi(E)$$

    yang jadi komponen kunci dalam perhitungan k-eigenvalue (k-eff).

6. Bentuk umum **fungsi 1D** (dipakai di xs, yield, dan semua komponen fission_energy_release)

    Semua kuantitas bergantung-energi di atas disimpan dalam salah satu dari bentuk berikut, ditandai atribut/type :

    | type | Bentuk data | Formula |
    |---|---|---|
    | constant | skalar tunggal | $y(E) = c$ |
    | Polynomial | array koefisien polinomial berurutan pangkat naik | $y(E) = \sum_{i=0}^{n} c_i E^i$ |
    | Tabulated1D | array 2×N (baris pertama nilai-x, baris kedua nilai-y), plus atribut breakpoints dan interpolation | interpolasi per segmen, lihat di bawah |
    | Sum | group berisi n sub-fungsi func_1..func_n | $y(x) = \sum_{k=1}^{n} f_k(x)$ — dipakai untuk reaksi redundant |

    Untuk Tabulated1D, tiap segmen antara titik $(x_i,y_i)$ dan $(x_{i+1},y_{i+1})$ diinterpolasi menurut kode ENDF (interpolation, per segmen yang dibatasi breakpoints):

    | Kode | Nama | Formula |
    |---|---|---|
    | 1 | Histogram | $y=y_i$, konstan sampai $x_{i+1}$ |
    | 2 | Lin-lin | $y = y_i + (y_{i+1}-y_i)\dfrac{x-x_i}{x_{i+1}-x_i}$ |
    | 3 | Lin-log | $y = y_i + (y_{i+1}-y_i)\dfrac{\ln(x/x_i)}{\ln(x_{i+1}/x_i)}$ |
    | 4 | Log-lin | $y = y_i\exp\!\Big[\ln\frac{y_{i+1}}{y_i}\cdot\frac{x-x_i}{x_{i+1}-x_i}\Big]$ |
    | 5 | Log-log | $y = y_i\exp\!\Big[\ln\frac{y_{i+1}}{y_i}\cdot\frac{\ln(x/x_i)}{\ln(x_{i+1}/x_i)}\Big]$ |

In [6]:
import h5py

# Ganti dengan nama file HDF5 Anda yang sebenarnya
filename = 'neutron/Ac225.h5' 

with h5py.File(filename, 'r') as f:
    # 1. Akses dataset xs
    xs_dataset = f['Ac225/reactions/reaction_017/294K/xs']
    
    # 2. Ambil nilai indeks threshold dari atribut
    # Gunakan .get() untuk menghindari error jika atribut tidak ada
    threshold_idx = xs_dataset.attrs.get('threshold_idx', 0) 
    print(f"Indeks Threshold: {threshold_idx}")
    
    # 3. (Opsional) Melihat energi ambang aktual (dalam eV)
    # Anda perlu mengakses grid energi yang berpasangan dengannya
    # Path energi di OpenMC biasanya ada di '/<isotop>/energy/<suhu>'
    energy_grid = f['Ac225/energy/294K'] 
    
    threshold_energy = energy_grid[threshold_idx]
    print(f"Energi Threshold (Fisis): {threshold_energy} eV")

Indeks Threshold: 945
Energi Threshold (Fisis): 12386100.0 eV


In [23]:
import h5py

filename = 'neutron/Ac225.h5' 

with h5py.File(filename, 'r') as f:
    # Mengecek atribut jenis partikel pada product_0
    partikel_0 = f['Ac225/reactions/reaction_016/product_0'].attrs.get('particle')
    print(f"Product 0 adalah: {partikel_0}")  # Hasilnya akan b'neutron'
    
    # Mengecek atribut jenis partikel pada product_1
    partikel_1 = f['Ac225/reactions/reaction_016/product_1'].attrs.get('particle')
    print(f"Product 1 adalah: {partikel_1}")  # Hasilnya akan b'photon'

Product 0 adalah: b'neutron'
Product 1 adalah: b'photon'


In [ ]:
nama_grup = 'Ac225/reactions/reaction_017/600K'
isi_folder = list(file_data[nama_grup].keys())

print(f"Isi di dalam folder {nama_grup}:")
print(isi_folder)

Isi di dalam folder Ac225/reactions/reaction_017/600K:
['xs']


In [ ]:
nama_dataset = 'Ac225/reactions/reaction_017/600K/xs'
data_list = file_data[nama_dataset][()].tolist()
print(data_list)
print(len(data_list))

[0.0, 0.0009224972, 0.00497209, 0.0510171, 0.1183766, 0.185736, 0.2821125, 0.378489, 0.495722, 0.612955, 0.712518, 0.812081, 0.9326105, 1.05314, 1.13875, 1.22436, 1.319235, 1.41411, 1.53592, 1.63591, 1.69214, 1.76384, 1.797061, 1.82248, 1.87087]
25


In [3]:
import h5py
import numpy as np

# Buka file photon
file_data = h5py.File('neutron/Ac225.h5', 'r')

def bongkar_total(nama_path, node):
    """
    Fungsi ini akan mengecek setiap node (baik berupa Folder/Group 
    maupun Dataset/File) dan mencetak atribut serta isi datanya.
    """
    print(f"\n{'='*60}")
    print(f"Path: {nama_path}")
    print(f"{'='*60}")
    
    # ---------------------------------------------------------
    # 1. MEMBACA ATRIBUT (Berlaku untuk Group maupun Dataset)
    # ---------------------------------------------------------
    print("[1] ATRIBUT:")
    if len(node.attrs) == 0:
        print("  --> Tidak ada atribut.")
    else:
        for key, val in node.attrs.items():
            print(f"  --> '{key}' = {val}")
            
    # ---------------------------------------------------------
    # 2. MEMBACA ISI DATA (Hanya jika node adalah Dataset)
    # ---------------------------------------------------------
    print("\n[2] ISI DATA:")
    if isinstance(node, h5py.Dataset):
        print(f"  --> Tipe Objek : Dataset (Ukuran: {node.shape}, Tipe: {node.dtype})")
        
        # Logika Skalar vs Array (seperti yang pernah kita bahas)
        if node.shape == (): 
            # Jika ukuran kosong (), ini adalah data skalar (1 angka)
            data_aktual = node[()]
            print(f"  --> Nilai Aktual : {data_aktual}")
            
        else:
            # Jika ini adalah array (1D, 2D, dst)
            data_aktual = node[:]
            
            if data_aktual.size == 0:
                print("  --> Data kosong.")
            elif data_aktual.size <= 10:
                print(f"  --> Seluruh Data : {data_aktual}")
            else:
                # Meratakan array (flatten) jika multidimensi agar mudah dicetak
                data_flat = data_aktual.flatten()
                print(f"  --> Total elemen : {data_aktual.size}")
                print(f"  --> 5 Data Awal  : {data_flat[:5]}")
                print(f"  --> 5 Data Akhir : {data_flat[-5:]}")
                
    elif isinstance(node, h5py.Group):
        # Jika node berupa Group, ia bertindak seperti folder dan tidak berisi angka langsung
        print("  --> Tipe Objek : Group (Bertindak sebagai Folder/Direktori)")

# Menjalankan fungsi ke SELURUH isi file
print("Memulai proses ekstrak file HDF5...")
file_data.visititems(bongkar_total)

Memulai proses ekstrak file HDF5...

Path: Ac225
[1] ATRIBUT:
  --> 'A' = 225
  --> 'Z' = 89
  --> 'atomic_weight_ratio' = 223.09
  --> 'metastable' = 0

[2] ISI DATA:
  --> Tipe Objek : Group (Bertindak sebagai Folder/Direktori)

Path: Ac225/energy
[1] ATRIBUT:
  --> Tidak ada atribut.

[2] ISI DATA:
  --> Tipe Objek : Group (Bertindak sebagai Folder/Direktori)

Path: Ac225/energy/0K
[1] ATRIBUT:
  --> Tidak ada atribut.

[2] ISI DATA:
  --> Tipe Objek : Dataset (Ukuran: (834,), Tipe: float64)
  --> Total elemen : 834
  --> 5 Data Awal  : [1.00000e-05 1.03125e-05 1.06250e-05 1.09375e-05 1.12500e-05]
  --> 5 Data Akhir : [18500000. 19000000. 19283260. 19500000. 20000000.]

Path: Ac225/energy/1200K
[1] ATRIBUT:
  --> Tidak ada atribut.

[2] ISI DATA:
  --> Tipe Objek : Dataset (Ukuran: (960,), Tipe: float64)
  --> Total elemen : 960
  --> 5 Data Awal  : [1.00000e-05 1.03125e-05 1.06250e-05 1.09375e-05 1.12500e-05]
  --> 5 Data Akhir : [18500000. 19000000. 19283260. 19500000. 20000000.]


# Photon

In [5]:
import h5py

file_data = h5py.File('photon/Ac.h5', 'r')

# Menampilkan seluruh hierarki/struktur di dalam file
print("Struktur isi file:")
file_data.visit(print)

Struktur isi file:
Ac
Ac/bremsstrahlung
Ac/bremsstrahlung/dcs
Ac/bremsstrahlung/electron_energy
Ac/bremsstrahlung/ionization_energy
Ac/bremsstrahlung/num_electrons
Ac/bremsstrahlung/photon_energy
Ac/coherent
Ac/coherent/anomalous_imag
Ac/coherent/anomalous_real
Ac/coherent/integrated_scattering_factor
Ac/coherent/scattering_factor
Ac/coherent/xs
Ac/compton_profiles
Ac/compton_profiles/J
Ac/compton_profiles/binding_energy
Ac/compton_profiles/num_electrons
Ac/compton_profiles/pz
Ac/energy
Ac/incoherent
Ac/incoherent/scattering_factor
Ac/incoherent/xs
Ac/pair_production_electron
Ac/pair_production_electron/xs
Ac/pair_production_nuclear
Ac/pair_production_nuclear/xs
Ac/photoelectric
Ac/photoelectric/xs
Ac/subshells
Ac/subshells/K
Ac/subshells/K/transitions
Ac/subshells/K/xs
Ac/subshells/L1
Ac/subshells/L1/transitions
Ac/subshells/L1/xs
Ac/subshells/L2
Ac/subshells/L2/transitions
Ac/subshells/L2/xs
Ac/subshells/L3
Ac/subshells/L3/transitions
Ac/subshells/L3/xs
Ac/subshells/M1
Ac/subshells/M

# Struktur File Photon Cross Section OpenMC (photon_dat.h5)

File yang Anda bedah ini adalah pustaka data interaksi foton OpenMC, yang secara internal disebut kelas `openmc.data.IncidentPhoton`. Setiap grup di tingkat pertama (Ac, Ag, Al, Am, dst.) adalah satu unsur, diberi nama sesuai simbol kimianya, dan isinya diambil dari sub-pustaka foto-atomik ENDF (MF=23 untuk cross section, MF=27 untuk faktor bentuk, MF=28 untuk relaksasi atom), digabung dengan data tambahan dari Seltzer & Berger (bremsstrahlung), NIST ESTAR (energi eksitasi rerata), dan Biggs, Mendelsohn & Mann via G4EMLOW (profil Compton).Photon interaction data is needed to run OpenMC with photon transport enabled, drawing bremsstrahlung cross sections from Seltzer and Berger, mean excitation energy from the NIST ESTAR database, and Compton profiles calculated by Biggs et al. and distributed via the Geant4 G4EMLOW data file.

**Catatan penting sebelum lanjut:** listing yang Anda tampilkan sepertinya hanya menangkap *group* dan *dataset* (misalnya lewat `h5py`'s `.visit()` atau `h5dump -n`), **tidak** menangkap *attribute*. Banyak informasi krusial — nomor atom Z, energi ikat & jumlah elektron per subshell, indeks ambang, daftar subshell yang tersedia, hingga energi eksitasi rerata untuk bremsstrahlung — disimpan sebagai **attribute**, bukan dataset terpisah, jadi "tidak kelihatan" di listing Anda meski benar-benar ada di file. Saya tandai eksplisit mana yang atribut vs dataset di bawah.

Konsep dasarnya: dataset **`xs`** = cross section total (barn) sebagai fungsi energi → menentukan *seberapa sering* interaksi ini terjadi. Dataset lain (`scattering_factor`, `J`, `transitions`, dst.) menentukan *bentuk diferensial* (sudut/energi keluaran) yang dipakai untuk sampling setelah jenis interaksi terpilih.

---

## 1. `Ac` (grup) dan atribut akarnya

| Item | Jenis | Isi |
|---|---|---|
| `(root).attrs['filetype']` | atribut file | `'data_photon'` |
| `(root).attrs['version']` | atribut file | `[major, minor]` versi format HDF5 |
| `Ac` | grup | seluruh data unsur Actinium |
| `Ac.attrs['Z']` | atribut | nomor atom (89 untuk Ac) |

## 2. `Ac/energy`

Grid energi **union/gabungan** dalam eV, hasil `numpy.union1d` dari seluruh grid energi native tiap reaksi (coherent, incoherent, photoelectric, pair production, dan semua subshell).Saat mengekspor ke HDF5, OpenMC menentukan union energy grid dari gabungan seluruh grid xs.x tiap reaksi, lalu menyimpannya sebagai dataset 'energy' Ini adalah grid referensi bersama: dataset `xs` di reaksi utama (coherent, incoherent, photoelectric, pair production) punya panjang **sama persis** dengan `energy` (dievaluasi di setiap titik). Dataset `xs` di tiap **subshell** lebih pendek karena baru dimulai dari indeks ambang (lihat §9).

## 3. `Ac/coherent` — Hamburan Koheren (Rayleigh), MT 502

| Item | Jenis | Satuan/Bentuk | Arti |
|---|---|---|---|
| `xs` | dataset | barn, sepanjang `energy` | cross section koheren total σ(E) |
| `scattering_factor` | dataset (Tabulated1D) | F(x,Z) vs x [Å⁻¹] | faktor bentuk atom |
| `integrated_scattering_factor` | dataset (Tabulated1D) | vs x² | integral kumulatif F²/Z² — tabel CDF siap-pakai |
| `anomalous_real` | dataset (Tabulated1D) | F′(E) vs energi foton | bagian riil faktor hamburan anomali (ENDF MT 506) |
| `anomalian_imag` *(anomalous_imag)* | dataset (Tabulated1D) | F″(E) vs energi foton | bagian imajiner (ENDF MT 505) |

Hamburan elastis foton bebas (Thomson) tidak bergantung energi:
$$\frac{d\sigma}{d\mu}\bigg|_{Thomson} = \pi r_e^2(1+\mu^2)$$

Untuk elektron terikat (Rayleigh), bentuknya dimodifikasi faktor bentuk plus koreksi anomali dekat *absorption edge*:persamaan lengkapnya adalah dσ/dμ = π r_e² (1+μ²) |F(x,Z) + F′ + iF″|², yang setara dengan π r_e² (1+μ²)[(F(x,Z)+F′(E))² + F″(E)²]

$$\frac{d\sigma(E,\mu)}{d\mu} = \pi r_e^2(1+\mu^2)\left[(F(x,Z)+F'(E))^2+F''(E)^2\right]$$

**Poin penting:** meskipun `anomalous_real`/`anomalous_imag` tersimpan di file, OpenMC mengabaikan hamburan anomali sehingga cross section diferensial yang benar-benar dipakai menjadi:

$$\frac{d\sigma(E,\mu)}{d\mu} = \pi r_e^2(1+\mu^2)\,F(x,Z)^2$$

Variabel transfer momentum:
$$x = ak\sqrt{1-\mu},\qquad k=\frac{E}{m_ec^2},\qquad a=\frac{m_ec^2}{\sqrt{2}\,hc}\approx 29.1\ \text{Å}^{-1}$$

Nilai maksimum $\bar{x}=ak\sqrt{2}$ dipakai membangun CDF ternormalisasi:
$$A(x^2,Z)=\int_0^{x^2}F(x,Z)^2\,dx^2$$

Inilah **persis** yang disimpan di `integrated_scattering_factor`— OpenMC pra-membangun tabel A(x²,Z) versus x² dan memakainya saat runtime untuk pencarian tabel pada fungsi distribusi kumulatif (dinormalisasi tambahan dengan /Z² di kode untuk kenyamanan numerik — tidak masalah karena hanya rasio A/A_maks yang dipakai). Algoritma sampling: cari $\bar{x}^2$ → ambil $A_{max}$ → tarik $\xi_1$, cari $x^2$ lewat pencarian biner pada tabel → hitung $\mu=1-2x^2/\bar{x}^2$ → terima/tolak dengan uji $\xi_2<(1+\mu^2)/2$ (rejection sampling Thomson).

## 4. `Ac/incoherent` — Hamburan Tak-Koheren (Compton), MT 504

| Item | Jenis | Arti |
|---|---|---|
| `xs` | dataset, barn | cross section Compton total |
| `scattering_factor` | dataset (Tabulated1D) | S(x,Z), fungsi hamburan tak-koheren |

Formula dasar untuk elektron bebas adalah Klein-Nishina:dσ_KN/dμ = π r_e² (k′/k)² [k′/k + k/k′ + μ² − 1], dengan k′ = k/(1+k(1−μ))

$$\frac{d\sigma_{KN}}{d\mu}=\pi r_e^2\left(\frac{k'}{k}\right)^2\left[\frac{k'}{k}+\frac{k}{k'}+\mu^2-1\right],\qquad k'=\frac{k}{1+k(1-\mu)}$$

Untuk elektron terikat, dikalikan faktor bentuk S(x,Z):
$$\frac{d\sigma}{d\mu}=\frac{d\sigma_{KN}}{d\mu}\,S(x,Z)$$

Sampling: gunakan metode rejection Kahn untuk k<3 dan metode langsung Koblinger untuk k≥3 pada Klein-Nishina, lalu lakukan rejection sampling tambahan terhadap S(x,Z)/S(x̄,Z).

## 5. `Ac/compton_profiles` — Pelebaran Doppler

| Item | Jenis | Bentuk | Arti |
|---|---|---|---|
| `pz` | dataset | 1D, satuan a.u. momentum ($\alpha m_ec$) | grid momentum bersama semua subshell |
| `J` | dataset | 2D `(n_subshell, n_pz)` | profil Compton $J_i(p_z)$ per subshell |
| `binding_energy` | dataset | eV, per subshell | energi ikat (bisa berbeda kecil dari attrs di §9) |
| `num_electrons` | dataset | per subshell | okupansi elektron |

Elektron terikat punya distribusi momentum, membatasi energi foton keluar maksimum dan melebarkan spektrumnya:energi keluar maksimum E′_max = E − E_b,i, sementara p_z = p·q/q adalah proyeksi momentum awal elektron pada arah momentum yang diserap foton

$$E'_{max}=E-E_{b,i}$$

$$p_z=\frac{E-E'-EE'(1-\mu)/m_ec^2}{-\alpha\sqrt{E^2+E'^2-2EE'\mu}}$$

$$J_i(p_z)=\iint \rho_i(\mathbf{p})\,dp_x\,dp_y$$

Faktor $\alpha$ (konstanta struktur halus) di penyebut adalah alasan `pz` bersatuan momentum atomik (a.u.), konsisten dengan konvensi tabel Hartree-Fock Biggs–Mendelsohn–Mann. Algoritma sampling:sampel μ dari persamaan hamburan tak-koheren, sampel subshell i memakai jumlah elektron per shell sebagai fungsi massa probabilitas, sampel p_z dari J_i(p_z) sebagai PDF, hitung E′ dari persamaan p_z, dan terima jika p_z < p_z,max untuk shell i.

Elektron recoil (Compton electron) yang tereksitasi:
$$E_-=E-E'-E_{b,i},\qquad \mu_-=\frac{E-E'\mu}{\sqrt{E^2+E'^2-2EE'\mu}}$$

## 6. `Ac/photoelectric` — Efek Fotolistrik, MT 522

Hanya satu dataset: `xs` (barn) — cross section **total**, langsung dari tabulasi ENDF MT522 (bukan hasil penjumlahan ulang subshell oleh OpenMC, walau secara fisis keduanya harus konsisten).

Elektron terpancar dari subshell $i$ dengan energi kinetik:
$$E_-=E-E_{b,i}$$

Shell tempat ionisasi terjadi dipilih sebagai variabel acak diskrit dengan fungsi massa probabilitas p_i = σ_pe,i/σ_pe, dengan σ_pe (total) adalah jumlah cross section tiap shell:

$$p_i=\frac{\sigma_{pe,i}}{\sigma_{pe}}$$

Ini sebabnya cross section per-subshell (§9) *harus* ada — dipakai sebagai bobot pemilihan shell. Sudut elektron fotolistrik disampel dari distribusi Sauter tingkat-K (elektron K paling dominan kontribusinya), $d\sigma_{pe}/d\mu_-\propto(1-\mu_-^2)/(1-\beta_-\mu_-)^4$ — detail implementasi ini tidak butuh data tambahan di file selain Z dan energi.Efek fotolistrik hanya mungkin terjadi bila energi foton melebihi energi ikat shell (edge energy), sehingga cross section yang menurun kontinu punya diskontinuitas berbentuk gigi gergaji di titik-titik ini, dan efek ini dominan pada energi rendah serta unsur berat.

## 7. `Ac/pair_production_electron` (MT 515) & `Ac/pair_production_nuclear` (MT 517)

Masing-masing hanya berisi `xs` (barn). Bedanya medan interaksi:

| Grup | MT | Medan | Ambang |
|---|---|---|---|
| `pair_production_nuclear` | 517 | inti atom | $2m_ec^2=1.022$ MeV |
| `pair_production_electron` | 515 | elektron atom (*triplet*) | $4m_ec^2=2.044$ MeV |

Rasio cross section triplet terhadap pair production dari inti kira-kira 1/Z, sehingga makin tidak penting untuk unsur Z tinggi. Cross section dasarnya Bethe-Heitler dengan koreksi Coulomb ($f_C$) dan fungsi screening ($\Phi_1,\Phi_2$):

$$\frac{d\sigma_{pp}}{d\epsilon}=\alpha r_e^2 Z^2\left[(\epsilon^2+(1-\epsilon)^2)(\Phi_1-4f_C)+\tfrac{2}{3}\epsilon(1-\epsilon)(\Phi_2-4f_C)\right],\quad \epsilon=\frac{E_-+m_ec^2}{E}$$

Sampling energi dan sudut pasangan elektron-positron memakai model semi-empiris Salvat (fungsi $\phi_1,\phi_2,\pi_1,\pi_2$ dan distribusi Sauter–Gluckstern–Hull untuk sudut) — semuanya dihitung dari Z dan energi foton saat runtime, **tidak** memerlukan dataset tambahan di luar `xs` yang sudah ada.

## 8. `Ac/subshells` — Data Relaksasi Atom (MF=28, MT 533) + Fotoionisasi per Subshell (MT 534–572)

Grup `subshells` punya atribut penting: **`designators`** (array string) — daftar subshell yang benar-benar tersedia untuk unsur ini.Atribut designators inilah yang dibaca kembali saat memuat data relaksasi atom dari HDF5

Setiap subshell (`K`, `L1`, …, `Q1`) adalah grup dengan:

| Item | Jenis | Arti |
|---|---|---|
| `.attrs['binding_energy']` | atribut, eV | energi ikat subshell |
| `.attrs['num_electrons']` | atribut | okupansi elektron saat netral |
| `xs` | dataset, barn | cross section fotoionisasi subshell ini |
| `xs.attrs['threshold_idx']` | atribut | indeks di `Ac/energy` tempat `xs` mulai |
| `transitions` | dataset (opsional) | tabel kaskade relaksasi (Auger + fluoresensi) |

**Soal `threshold_idx` — ini krusial untuk pemetaan data yang benar:** cross section subshell hanya tak-nol mulai energi ≥ binding energy, jadi OpenMC mencari indeks pada grid union tempat threshold subshell berada, menginterpolasi cross section subshell pada energy[idx:], menuliskannya sebagai dataset xs, lalu mencatat idx sebagai atribut threshold_idx. Artinya: **`xs[j]` berpasangan dengan `energy[threshold_idx + j]`**, bukan `energy[j]`. Kalau Anda plot tanpa memperhitungkan ini, kurva Anda akan salah geser.

**Kenapa O4, O5, P1–P5, Q1 tidak punya `transitions`?** Dataset transitions hanya ditulis jika shell tersebut memang punya entri di tabel transisi — subshell terluar/valensi sering tak punya data relaksasi terukur di evaluasi ENDF. Menariknya, pola subshell yang muncul untuk Ac (…O1–O5, P1–P5, Q1, **tanpa** O6+ atau P6+ atau Q2+) cocok persis dengan konfigurasi elektron keadaan-dasar Ac ($Z=89$, $[\text{Rn}]6d^17s^2$) — subshell 5f, 6f, 7p yang kosong pada atom netral memang tidak ditabulasi.

**Struktur `transitions`** (4 kolom, semua disimpan sebagai float): `[secondary, tertiary, energy(eV), probability]`. Kolom *secondary*/*tertiary* adalah **indeks numerik** ke tabel subshell berikut (bukan string):

```
0=None  1=K   2=L1  3=L2  4=L3  5=M1  6=M2  7=M3  8=M4  9=M5
10=N1  11=N2 12=N3 13=N4 14=N5 15=N6 16=N7 17=O1 18=O2 19=O3
20=O4  21=O5 22=O6 23=O7 24=O8 25=O9 26=P1 27=P2 28=P3 29=P4
30=P5  31=P6 32=P7 33=P8 34=P9 35=P10 36=P11 37=Q1 38=Q2 39=Q3
```

Setiap baris = satu jalur pengisian vacancy di subshell ini: elektron datang dari shell *secondary*. Jika *tertiary*=0 (None) → **transisi radiatif** (foton fluoresensi), energinya:
$$E_{fluor}=E_{b,vacancy}-E_{b,secondary}$$
Jika *tertiary*≠0 → **transisi non-radiatif** (elektron Auger keluar dari shell *tertiary*, meninggalkan vacancy baru di sana):
$$E_{Auger}=E_{b,vacancy}-E_{b,secondary}-E_{b,tertiary}$$

Kolom *probability* adalah probabilitas cabang (branching ratio) individual tiap jalur — dipakai langsung sebagai fungsi massa probabilitas untuk memilih transisi mana yang terjadi, lalu proses berulang secara rekursif untuk vacancy baru yang tercipta hingga tak ada transisi tersisa.

## 9. `Ac/bremsstrahlung` — Data Elektron Sekunder (TTB)

Ini bukan interaksi foton langsung, tapi data pendukung untuk memodelkan foton bremsstrahlung dari elektron sekunder (hasil Compton/fotolistrik/produksi pasangan) tanpa transport elektron penuh.

| Item | Jenis | Bentuk | Arti |
|---|---|---|---|
| `.attrs['I']` *(tersembunyi)* | atribut, eV | energi eksitasi rerata unsur |
| `dcs` | dataset | 2D `(200, k)`, barn | cross section terskala $\chi(Z,T,\kappa)$ |
| `electron_energy` | dataset | 1D, 200 titik, eV | grid $T$: `np.logspace(3,9,200)` = 1 keV–1 GeV |
| `photon_energy` | dataset | 1D, k titik, tak-berdimensi (0–1) | grid $\kappa=E/T$ |
| `num_electrons` | dataset | per subshell | untuk hitung *density effect* (bisa negatif = elektron konduksi) |
| `ionization_energy` | dataset | eV, per subshell | untuk hitung *density effect* |

Grid electron_energy dibangun ulang secara tetap sebagai 200 titik logaritmik dari 10³ hingga 10⁹ eV, lalu dcs asli (dari tabel Seltzer-Berger dalam milibarn) diinterpolasi-spline kubik dalam log-energi ke grid baru ini dan dikonversi ke barn. `num_electrons`/`ionization_energy` di sini **berbeda sumber** dari atribut subshell di §9 — keduanya dibaca dari file pendukung terpisah (`density_effect.h5`, basis ESTAR/Sternheimer), jadi jumlah entrinya bisa tidak sama dengan daftar `designators`.

Cross section bremsstrahlung diferensial:dσ_br/dE = (Z²/β²)(1/E)χ(Z,T,κ), dengan κ=E/T adalah energi foton terreduksi dan χ adalah cross section terskala yang diukur secara eksperimental — **inilah persis isi dataset `dcs`**.

$$\frac{d\sigma_{br}}{dE}=\frac{Z^2}{\beta^2}\frac{1}{E}\chi(Z,T,\kappa)$$

Daya henti radiatif diperoleh integrasi `dcs` atas seluruh `photon_energy`:
$$S_{rad}(T)=n\frac{Z^2}{\beta^2}T\int_0^1\chi(Z,T,\kappa)\,d\kappa$$

Daya henti tumbukan (Bethe), memakai `I`:
$$S_{col}(T)=\frac{2\pi r_e^2m_ec^2}{\beta^2}N_A\frac{Z}{A_M}\left[\ln(T^2/I^2)+\ln(1+\tau/2)+F(\tau)-\delta_F(T)\right]$$

Efek densitas $\delta_F$ inilah tempat `num_electrons` ($f_i=n_i/Z$) dan `ionization_energy` (sebagai proksi frekuensi oscillator $h\nu_i$) dipakai, diselesaikan self-consistent lewat metode Sternheimer:δ_F(β) = Σ f_i ln[(l_i²+l²)/l_i²] − l²(1−β²), dengan f_i=n_i/Z, dan faktor ρ ditentukan agar konsisten dengan I lewat aturan sum Bethe

$$\delta_F(\beta)=\sum_i f_i\ln\!\left[\frac{l_i^2+l^2}{l_i^2}\right]-l^2(1-\beta^2)$$

Ini kemudian dipakai membangun rentang CSDA dan hasil-kali foton pada pendekatan TTB:rentang CSDA R(T)=∫dT′/S(T′), dan hasil-kali foton bremsstrahlung Y(T,E_cut) dibangun dari jalur bebas rata-rata terbalik λ_br⁻¹ yang mengintegralkan dcs dari κ_cut sampai 1, lalu photon energy disampel dari CDF tertabulasi memakai metode komposisi pada grid energi log. *(Catatan sejarah: publikasi validasi awal OpenMC—2018—mencatat efek densitas belum diterapkan saat itu; dokumentasi metode terkini menyertakan formulanya secara penuh seperti di atas.)*

---

## 10. Konvensi Umum

- **Satuan**: energi selalu eV, cross section selalu barn, `photon_energy` bremsstrahlung tak-berdimensi (rasio 0–1).
- **Interpolasi**: begitu dibaca ulang lewat `openmc.data`, semua dataset `xs` direkonstruksi sebagai `Tabulated1D` dengan hukum interpolasi **log-log** (kode ENDF `5`) terhadap grid `energy` bersama — info skema interpolasi asli tidak disimpan ulang per-titik untuk `xs`.Ini terlihat dari PhotonReaction.from_hdf5 yang selalu memaksa Tabulated1D(energy, xs, [len(xs)], [5]) Sebaliknya, `scattering_factor`/`anomalous_real`/`anomalous_imag` menyimpan skema interpolasinya sendiri (pola kelas `Tabulated1D`: pasangan x,y plus atribut breakpoints/interpolation).

Perhatikan: `IncidentPhoton.from_hdf5('photon_dat.h5')` (memberi nama file langsung) hanya membaca grup **pertama** dalam file — untuk file gabungan berisi Ac, Ag, Al, Am, dst., Anda harus membuka file dengan `h5py` lalu mengoper `Group` unsur yang diinginkan seperti contoh di atas.

Referensi lengkap: [Photon Physics](https://docs.openmc.org/en/stable/methods/photon_physics.html), [Charged Particle Physics](https://docs.openmc.org/en/stable/methods/charged_particles_physics.html), dan [source code `openmc/data/photon.py`](https://docs.openmc.org/en/stable/_modules/openmc/data/photon.html) untuk ground-truth struktur HDF5.

In [9]:
import h5py
import numpy as np

# Buka file photon
file_data = h5py.File('photon/Ac.h5', 'r')

def bongkar_total(nama_path, node):
    """
    Fungsi ini akan mengecek setiap node (baik berupa Folder/Group 
    maupun Dataset/File) dan mencetak atribut serta isi datanya.
    """
    print(f"\n{'='*60}")
    print(f"Path: {nama_path}")
    print(f"{'='*60}")
    
    # ---------------------------------------------------------
    # 1. MEMBACA ATRIBUT (Berlaku untuk Group maupun Dataset)
    # ---------------------------------------------------------
    print("[1] ATRIBUT:")
    if len(node.attrs) == 0:
        print("  --> Tidak ada atribut.")
    else:
        for key, val in node.attrs.items():
            print(f"  --> '{key}' = {val}")
            
    # ---------------------------------------------------------
    # 2. MEMBACA ISI DATA (Hanya jika node adalah Dataset)
    # ---------------------------------------------------------
    print("\n[2] ISI DATA:")
    if isinstance(node, h5py.Dataset):
        print(f"  --> Tipe Objek : Dataset (Ukuran: {node.shape}, Tipe: {node.dtype})")
        
        # Logika Skalar vs Array (seperti yang pernah kita bahas)
        if node.shape == (): 
            # Jika ukuran kosong (), ini adalah data skalar (1 angka)
            data_aktual = node[()]
            print(f"  --> Nilai Aktual : {data_aktual}")
            
        else:
            # Jika ini adalah array (1D, 2D, dst)
            data_aktual = node[:]
            
            if data_aktual.size == 0:
                print("  --> Data kosong.")
            elif data_aktual.size <= 10:
                print(f"  --> Seluruh Data : {data_aktual}")
            else:
                # Meratakan array (flatten) jika multidimensi agar mudah dicetak
                data_flat = data_aktual.flatten()
                print(f"  --> Total elemen : {data_aktual.size}")
                print(f"  --> 5 Data Awal  : {data_flat[:5]}")
                print(f"  --> 5 Data Akhir : {data_flat[-5:]}")
                
    elif isinstance(node, h5py.Group):
        # Jika node berupa Group, ia bertindak seperti folder dan tidak berisi angka langsung
        print("  --> Tipe Objek : Group (Bertindak sebagai Folder/Direktori)")

# Menjalankan fungsi ke SELURUH isi file
print("Memulai proses ekstrak file HDF5...")
file_data.visititems(bongkar_total)

Memulai proses ekstrak file HDF5...

Path: Ac
[1] ATRIBUT:
  --> 'Z' = 89

[2] ISI DATA:
  --> Tipe Objek : Group (Bertindak sebagai Folder/Direktori)

Path: Ac/bremsstrahlung
[1] ATRIBUT:
  --> 'I' = 841.0

[2] ISI DATA:
  --> Tipe Objek : Group (Bertindak sebagai Folder/Direktori)

Path: Ac/bremsstrahlung/dcs
[1] ATRIBUT:
  --> Tidak ada atribut.

[2] ISI DATA:
  --> Tipe Objek : Dataset (Ukuran: (200, 30), Tipe: float64)
  --> Total elemen : 6000
  --> 5 Data Awal  : [0.00048215 0.00050415 0.00052781 0.00055315 0.00057995]
  --> 5 Data Akhir : [0.00232825 0.00161003 0.00149357 0.00144237 0.0014264 ]

Path: Ac/bremsstrahlung/electron_energy
[1] ATRIBUT:
  --> Tidak ada atribut.

[2] ISI DATA:
  --> Tipe Objek : Dataset (Ukuran: (200,), Tipe: float64)
  --> Total elemen : 200
  --> 5 Data Awal  : [1000.         1071.89131921 1148.95100019 1231.55060329 1320.08840083]
  --> 5 Data Akhir : [7.57525026e+08 8.11984499e+08 8.70359136e+08 9.32930403e+08
 1.00000000e+09]

Path: Ac/bremsstrah